## README - how to adapt the code for a new environment
This jupyter shows how the offline verification looks like to provide you context.

To adapt it for new environment, only needs:
1) A function to build the data to be verified by the LLM
2) A base system prompt containing any environment-specific relevant info
 - That is, a trace of execution in `images`, `text`, and the `objective` the generator is trying to accomplish
3) Possibly small changes to the first-pass prompt and evaluation criterias

I can do (2) and (3) relatively fast, but conditioned on how the data in (1) will look like to parse it.

With this during verification, the LLM receives:
- input: `[sys_prompt] [the objective] [execution trace] <first_pass response> [eval_criteria] [response_format]`
- output: LLM generation that must abide to the `response_format`

Below is example is based on the VWA environment. Relevant outputs are in the jupyter cells and the folders within `verify_example`.
- I didn't this runnable as it depends on many VWA-specific stuff.

In [ ]:
from offline_experiments.vwa_specific import get_intent_message, get_trace_data
from llms.llm_utils import call_llm, visualize_prompt
from offline_experiments.build_prompt import safe_format
this_file_folder = "./offline_experiments/verify_example"

## Prompt Parts

Run the cell below or open the `full_prompt.html` example to see the final prompt given to the LLM.

To preserve consistency across experiments, it's built by composing pieces to form: `[system prompt]` `[data to be verified]` `[query requesting verification]`

For verification query for example, it's composed by `[evaluation criteria]` `[CoT parts]` `[response format]`.

Continue reading for an example of these parts and how they're composed.

#### System Prompt

VWA system prompt is built by: a (i) base system prompt template where (ii) some additional information is added based on the experiment considered (see the {{}} placeholders below)

You need to care less about the later  because there aren't many ablations for the other envs.

In [23]:
# --- System prompt ---

sys_prompt_critic_base = f"""You are an intelligent agent tasked with supervising an assistant navigating a web browser to accomplish a web-based task. Your job is to evaluate the assistant's work, and provide feedback so it can progress towards the objective.
\n
## Here's the information you'll have:
### The objective: This is the task the assistant is trying to complete.
### Webpage screenshots: These are screenshots of the webpage, with each interactable element assigned a unique numerical id. Each bounding box and its respective id shares the same color.
{{trace_info}}{{summary_info}}{{web_knowledge_info}}
\n
## Assistant's capabilities: To effectively analyze the assistant's work, consider the actions it can perform. These actions fall into the following categories:
### Page Operation Actions:
```click [id]```: Click on an element with a specific id on the webpage.
```type [id] [content] [enter_after]```: Type the content into the field with id. If `enter_after` is 0, the "Enter" key is not pressed after typing; otherwise, the "Enter" key is automatically pressed.
```hover [id]```: Hover over an element with id.
```press [key_comb]```: Press a keyboard key or key combination (e.g., delete, ctrl+a).
```scroll [down]``` or ```scroll [up]```: Scroll the webpage up or down.

### Tab Management Actions:
```new_tab```: Open a new, empty browser tab.
```tab_focus [tab_index]```: Switch the browser's focus to a specific tab using its index.
```close_tab```: Close the currently active tab.

### URL Navigation Actions:
```goto [url]```: Navigate to a specific URL.
```go_back```: Navigate to the previously viewed page.
```go_forward```: Navigate to the next page (if a previous 'go_back' action was performed).

### Completion Action:
```stop [answer]```: Issue this action when you believe the task is complete. If the objective is to find a text-based answer, provide the answer in the bracket. If you believe the task is impossible, issue this action with optionally a reason why.
{{privileged_hint}}\n
## To be successful, it is very important to follow the following rules:
1. You must not not assume the assistant's work is correct or incorrect beforehand. You should come up with your own opinion based on the information provided.
2. You must connect the dots between the objective and the information provided.{{web_knowledge_rule}}
"""

# Examples of additional info provided depending on the experiment:
### When the execution trace includes the generator's VERBAL responses, I add this:
trace_info_utt = "\n### The execution trace: This is a sequence of webpage screenshots paired with the assistant's responses, detailing the web navigation so far."

### When the overall verification includes a first pass, I add this:
web_knowledge_info = "\n### General web knowledge: This is a general description of how tasks like this are typically accomplished on the web."
web_knowledge_rule = "\n3. Use the General web knowledge as a guide, but also consider the context of the specific task given to you."

# The final sys_prompt is created as follows:
sys_prompt = safe_format(sys_prompt_critic_base, web_knowledge_rule=web_knowledge_rule, trace_info=trace_info_utt, web_knowledge_info=web_knowledge_info)
print(sys_prompt)

You are an intelligent agent tasked with supervising an assistant navigating a web browser to accomplish a web-based task. Your job is to evaluate the assistant's work, and provide feedback so it can progress towards the objective.


## Here's the information you'll have:
### The objective: This is the task the assistant is trying to complete.
### Webpage screenshots: These are screenshots of the webpage, with each interactable element assigned a unique numerical id. Each bounding box and its respective id shares the same color.

### The execution trace: This is a sequence of webpage screenshots paired with the assistant's responses, detailing the web navigation so far.
### General web knowledge: This is a general description of how tasks like this are typically accomplished on the web.


## Assistant's capabilities: To effectively analyze the assistant's work, consider the actions it can perform. These actions fall into the following categories:
### Page Operation Actions:
```click [i

The final query is built by composing: 
`[evaluation criteria]` `[CoT parts]` `[response format]`

Other envs don't need many of these parts as there are less ablations.

In [55]:

# ---- Evaluation Criteria ----
# This is the evaluation criteria for the case with 3 outcomes.
eval_criteria_tri = """
SUCCESS: The assistant executed **all of** what's necessary to complete the objective. The task is fully accomplished.
PARTIAL SUCCESS: The assistant executed **most of** what's necessary to complete the objective. The task is partially accomplished.
FAILURE: The assistant executed **mostly incorrect** steps. The task is not accomplished, and major revisions are needed.
"""

# ---- CoT parts ----
# Some intermediate reasoning before verification. Two examples:
basic_cot = """
REASONING: [Step by step reasoning to come up with your evaluation and feedback]
"""

desc_cot = """
EXECUTION TRACE DESCRIPTION: [Understand and describe the assistant's work]
REASONING: [Step by step reasoning to come up with your evaluation and feedback]
"""

# ---- Final query ----
# This is the final query to be given to the LLM. It includes the evaluation criteria, the CoT parts, and the response format.

response_format_template = f"""
{{cot_parts}}
EVALUATION: [Your evaluation following the evaluation criteria]
FEEDBACK: [Feedback so the assistant can progress towards the objective]"""

eval_prompt_template = f"""Now please provide your response.
\n
## Here is the evaluation criteria:
{{eval_criteria}}
\n
## Provide your response as follows:
{{response_format}}
"""

# obs.: the .strip() is used to remove the leading and trailing newlines I added to prompt parts for readability.
response_format = safe_format(response_format_template, cot_parts=basic_cot.strip()).strip()
eval_prompt = safe_format(eval_prompt_template, eval_criteria=eval_criteria_tri.strip(), response_format=response_format).strip()
print(eval_prompt)

Now please provide your response.


## Here is the evaluation criteria:
SUCCESS: The assistant executed **all of** what's necessary to complete the objective. The task is fully accomplished.
PARTIAL SUCCESS: The assistant executed **most of** what's necessary to complete the objective. The task is partially accomplished.
FAILURE: The assistant executed **mostly incorrect** steps. The task is not accomplished, and major revisions are needed.


## Provide your response as follows:
REASONING: [Step by step reasoning to come up with your evaluation and feedback]
EVALUATION: [Your evaluation following the evaluation criteria]
FEEDBACK: [Feedback so the assistant can progress towards the objective]


## Example of second-pass / verification

The code has some config definitins which are based on the type of experiment

In [ ]:
config = {
    "prompt_args": {
        "eval_criteria": "tri",  # this uses the criteria with SUCCESS, PARTIAL SUCCESS, FAILURE
        "cot_part": "basic_cot",  # this includes [REASONING] part in the verifier query
        "trace_info": "utt",  # when building the execution trace, provides the utterances of the Agent during the execution
        "add_summ_info": False, # not used anymore
        "add_expectation_info": True, # Adds to system prompt info that it will receive first pass response
        "k_config": [{"expert": True, "conditional": False, "cached_k_dir": None},], # Config of the first pass response (verification.py needs to know too to build the prompt)
    },
    "out_dir": this_file_folder,
}

1) Retrieve the execution trace data to be verified.

In [61]:
# Traces in VWA are stored as HTMLs like: folder/<domain>/render_{task_id}.html
domain, task_id = "shopping", 37
trace_path = f"{this_file_folder}/vwa_traces/{domain}/render_{task_id}.html"
trace_data = get_trace_data(trace_path, task_id)  # This retrieves everything relevant from an execution from the HTML logtrace_data

{'objective': {'text': 'Add the navy blue one in the second column to my wish list.',
  'images': []},
 'trajectory': <utils.trajectory_view.TrajectoryView at 0x7fb810e4aa10>,
 'meta_data': {'action_str_history': ['None',
   'click [37] where [37] is [A] element with content [Add to Wish List]',
   'stop [The navy blue blanket has been added to your wish list.]']},
 'task_id': '37',
 'trajectory_path': '/home/mashalimay/webarena/modular_agent/offline_experiments/verify_example/vwa_traces/shopping/render_37.html'}

This is how it looks like for VWA, but this will probably change across environments.

Important components to have:
- `objective`: this is the description of what the generator that generated the trajectory was trying to accomplish. Can have `text` and `images`.
- The `trajectory` object, which is basically a sequence of `[image]` `[text]` detailing the execution, the same as in the `full_prompt.html`.

In [62]:
trace_data

{'objective': {'text': 'Add the navy blue one in the second column to my wish list.',
  'images': []},
 'trajectory': <utils.trajectory_view.TrajectoryView at 0x7fb810e4aa10>,
 'meta_data': {'action_str_history': ['None',
   'click [37] where [37] is [A] element with content [Add to Wish List]',
   'stop [The navy blue blanket has been added to your wish list.]']},
 'task_id': '37',
 'trajectory_path': '/home/mashalimay/webarena/modular_agent/offline_experiments/verify_example/vwa_traces/shopping/render_37.html'}

2) Build the prompt

The prompt can be built simply like:

```python
prompt = [
    {"role": "system", "content": sys_prompt}, # built above
    "## Objective: Place the ball on the red table", #  from get_trace_data
    Image of the ball at initial position, # from get_trace_data
    "To place the ball I will ....", # agent utterance, from get_trace_data. Might not have it.
    Image of the ball at final position, # from get_trace_data
    eval_prompt # built above
]
```

VWA has some other quirks so the below uses auxiliary functions to build a thing like the above. Don't bother about them

In [ ]:
# 2) Build the prompt
# This builds the trajectory executed by the Agent, which will be evaluated
# 
from offline_experiments.utils_offline_exper import get_trajectory_msgs
trajectory_msgs = get_trajectory_msgs(config, trace_data) 

# This builds the definition of the task the generator was trying to accomplish during the execution above.
# I have a function because VWA has some quirks.
# Can be simply a string like: intent = "Place the ball over the red square"
intent = get_intent_message(trace_data=trace_data) 


Here is how trajectory_msgs and intent looks like. 
- Note: It's in format used by the `llms` library but can be raw text/images/videos.

In [64]:
trajectory_msgs

[Message(role='user', contents=[ContentItem(type='text', data='## Here is the trace of execution so far:\n', meta_data={}, id=None)], name='user', meta_data={}),
 Message(role='user', contents=[ContentItem(type='text', data='### STATE `t-2` SCREENSHOT:', meta_data={}, id=None), ContentItem(type='image', data=<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1280x2048 at 0x7FB8159EE090>, meta_data={'img_detail': 'auto'}, id=None)], name='user', meta_data={}),
 Message(role='assistant', contents=[ContentItem(type='text', data='Let\'s think step-by-step. The objective is to add the navy blue blanket, which is located in the second column, to the wish list. From the screenshot, the blanket in question is the "PEACE NEST Lightweight Down and Feather Fiber Throw Blanket" with a price of $39.99 and its ID for adding to the wish list is [37]. \nIn summary, the next action I will perform is ```click [37]```.', meta_data={}, id=None)], name='assistant', meta_data={}),
 Message(role='user', c

In [65]:
intent

[Message(role='user', contents=[ContentItem(type='text', data='## OBJECTIVE:\nAdd the navy blue one in the second column to my wish list.', meta_data={}, id=None)], name='', meta_data={})]

Suppose in the first pass, LLM produced a response like below.

In [68]:
k = """To add an item to your wish list on a typical e-commerce website, follow these general steps:
1. **Locate the Item**: Find the item you want to add to your wish list. In this case, it's the navy blue throw in the second column.
2. **Select the Item**: Click on the item or its title to view more details. This often takes you to the product's dedicated page.
3. **Add to Wish List**: Look for an option to add the item to your wish list. This is usually a button or link labeled "Add to Wish List" or a heart icon. Click on it.
4. **Confirm Addition**: Some websites may ask you to confirm the addition or provide a notification that the item has been successfully added to your wish list.
5. **View Wish List**: You can usually view your wish list by navigating to your account section or a dedicated wish list page on the website.
These steps may vary slightly depending on the website's design and functionality."""

Then the full verifier prompt is:

In [74]:
# A little template to "inject" this `k` into the final prompt.
k_injection = """## General web knowledge:
{k}"""

full_prompt = [
    {"role": "system", "content": sys_prompt},
    intent,
    trajectory_msgs,
    k_injection.format(k=k),
    eval_prompt,
]
from llms.llm_utils import visualize_prompt
visualize_prompt(full_prompt, output_path=f"{this_file_folder}/full_prompt.html", verbose=True)

Conversation saved to /home/mashalimay/webarena/modular_agent/offline_experiments/verify_example/full_prompt.html


**IMPORTANT NOTE**: where `k` comes from:

To save money and time, I first run the `first-pass.py` and store `k` outputs like the above.

Then during `verification.py` any ablations of the verifier get's the prompts from these caches.

## LLM verifier call

Some configs for the LLM

In [82]:

gen_config = {
    "model": "gemini-2.5-flash-preview-04-17",
    "temperature": 1.0,
    "max_tokens": 8192,
    "top_p": 0.5,
    "top_k": 40,
    "response_format": None,
    "seed": 0,
}

The code will define the output directory to store the HTML trace and token usage data of the experiment. 

It does something like below:

In [90]:
from offline_experiments.config_run import build_config_name
config_name = build_config_name(
    eval_criteria=config["prompt_args"]["eval_criteria"],
    cot_part=config["prompt_args"]["cot_part"],
    trace_info=config["prompt_args"]["trace_info"],
    k_config=config["prompt_args"]["k_config"][0],
)

destination_dir = f"{config['out_dir']}/{config_name}"
print(destination_dir)

/home/mashalimay/webarena/modular_agent/offline_experiments/verify_example/tri-basic_cot-utt-uncond-expert


In [91]:
api_responses, model_messages = call_llm(
    gen_kwargs=gen_config,
    prompt=full_prompt,
    conversation_dir=f"{destination_dir}/conversation", # all HTML traces of the conversation will be here
    usage_dir=f"{destination_dir}/usage", # all token usage data of the conversation will be here
    call_id = task_id,
    verbose=True,
)

CALLING MODEL: `gemini-2.5-flash-preview-04-17`: generating 1 outputs...


Conversation saved to /home/mashalimay/webarena/modular_agent/offline_experiments/verify_example/tri-basic_cot-utt-uncond-expert/conversation/37.html
Conversation saved to /home/mashalimay/webarena/modular_agent/offline_experiments/verify_example/tri-basic_cot-utt-uncond-expert/conversation/37.txt


In [89]:
print(model_messages[0].text())

REASONING: The objective was to add the navy blue blanket in the second column to the wish list. The assistant correctly identified the navy blue blanket and clicked the "Add to Wish List" button associated with it. The subsequent screenshot shows that the item was successfully added to the wish list. Therefore, the objective has been fully accomplished.
EVALUATION: SUCCESS
FEEDBACK: The assistant successfully completed the task.
